# Assignment 07 - Notebook 02
## RAG từ đầu với BM25

Notebook trình bày chuỗi Indexing → Retrieval → Augmentation → Generation bằng code tự viết, không phụ thuộc API trả phí.

## 1. Khởi tạo và đọc corpus

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Project:', ROOT)

Project: C:\DATA\assignment3\assignment7


In [2]:
documents = json.loads((ROOT/'knowledge/documents.json').read_text(encoding='utf-8'))
[(d['id'], d['title']) for d in documents]

[('who-overview', 'WHO - Diabetes overview'), ('cdc-risk', 'CDC - Type 2 diabetes risk factors'), ('cdc-symptoms', 'CDC - Diabetes symptoms'), ('niddk-tests', 'NIDDK - Diabetes tests and diagnosis'), ('who-prevention', 'WHO - Prevention and management'), ('model-limit', 'Giới hạn mô hình Assignment 07')]

## 2. Chunking và indexing

Corpus mẫu đã được biên tập thành sáu đoạn ngắn. Với PDF dài, cần chia theo tiêu đề hoặc cửa sổ token có overlap để tránh cắt mất ngữ nghĩa.

In [3]:
for d in documents:
    print(d['id'], '->', len(d['text'].split()), 'từ')

who-overview -> 69 từ
cdc-risk -> 46 từ
cdc-symptoms -> 44 từ
niddk-tests -> 50 từ
who-prevention -> 52 từ
model-limit -> 52 từ


## 3. BM25 scratch

BM25 dùng tần suất từ, độ hiếm của từ trong corpus và chuẩn hóa độ dài tài liệu. Nó là baseline nhanh, dễ giải thích nhưng không hiểu đầy đủ đồng nghĩa/ngữ cảnh.

In [4]:
from src.rag_agent import BM25Retriever, tokenize
retriever = BM25Retriever(documents)
print('Kích thước từ vựng:', len(retriever.idf))

Kích thước từ vựng: 189


## 4. Retrieval Top-K

Chỉ các đoạn có điểm lớn hơn 0 được đưa vào context, giúp giảm nhiễu so với đưa toàn bộ tài liệu vào prompt.

In [5]:
query = 'Yếu tố nguy cơ của đái tháo đường type 2 là gì?'
hits = retriever.search(query, top_k=3)
[(h['title'], h['score']) for h in hits]

[('CDC - Type 2 diabetes risk factors', 10.6679), ('WHO - Diabetes overview', 5.3082), ('NIDDK - Diabetes tests and diagnosis', 2.4163)]

## 5. Augmentation và generation

Bản offline ghép các bằng chứng truy xuất và giữ URL nguồn. Khi có LLM, chính context này được đưa vào prompt với yêu cầu chỉ trả lời dựa trên nguồn.

In [6]:
context = '\n'.join(f"[{i+1}] {h['text']}" for i,h in enumerate(hits))
print(context)

[1] Các yếu tố liên quan nguy cơ đái tháo đường type 2 gồm thừa cân hoặc béo phì, ít vận động, tiền sử gia đình, tuổi cao hơn, tăng huyết áp và tiền đái tháo đường. Nguy cơ thống kê không phải chẩn đoán cá nhân.
[2] Đái tháo đường là bệnh mạn tính xảy ra khi tuyến tụy không sản xuất đủ insulin hoặc cơ thể không sử dụng hiệu quả insulin. Tăng đường huyết kéo dài có thể gây tổn thương tim, mạch máu, mắt, thận và thần kinh. Type 1 liên quan thiếu insulin và cần dùng insulin; type 2 thường liên quan tình trạng cơ thể sử dụng insulin không hiệu quả.
[3] Các xét nghiệm thường dùng để đánh giá đái tháo đường gồm HbA1c, glucose huyết tương lúc đói, nghiệm pháp dung nạp glucose đường uống và glucose huyết tương ngẫu nhiên khi có triệu chứng. Bác sĩ cần xác nhận và diễn giải kết quả trong bối cảnh lâm sàng.


## 6. Cải tiến RAG

Có thể thêm query rewriting, query expansion, decomposition trước retrieval; sau retrieval dùng reranking và context compression. GraphRAG bổ sung các đường đi giữa thực thể để xử lý câu hỏi multi-hop.